# Notebook 02 — PConv U-Net Training

**Purpose:** Train the Partial Convolution U-Net on the WikiArt inpainting
task using Colab T4 (free) or Kaggle P100 (free, 30h/week).

**Resume safety:** All state is on Google Drive — checkpoints, logs,
validation figures.  When Colab disconnects, just re-run cells 1–5 from
the top.  The Trainer auto-detects the latest checkpoint and resumes.

**Cells:**
1. Setup — packages, Drive mount, code from GitHub (or Drive cache)
2. GPU verification — fail fast if CPU runtime
3. Config — display + override Drive paths
4. Initialize Trainer — auto-resume detection
5. Train — single call, runs the full loop
6. Results — plot training curves + show best validation samples

## Cell 1 — Setup

Install dependencies, mount Drive, fetch the project source.

On the **first run**, this clones the private repo from GitHub (you'll be
prompted for a Personal Access Token with `repo` scope) and caches a copy
to Drive.  On every subsequent run, the code is restored from the Drive
cache — no GitHub auth needed.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  Set this to True once after Claude pushes new code to GitHub.
#  Set back to False on subsequent runs to use the fast Drive cache.
# ════════════════════════════════════════════════════════════════════════════
REFRESH_FROM_GITHUB = True

# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q torch torchvision albumentations pyyaml tqdm pandas \
               matplotlib Pillow opencv-python scikit-learn \
               'torchmetrics[image]' lpips

# ── Mount Drive ──────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ────────────────────────────────────────────────────────────────────
import os
import shutil
import sys
from pathlib import Path

DRIVE_ROOT  = Path('/content/drive/MyDrive/art_restoration')
DRIVE_CODE  = DRIVE_ROOT / 'code'                # cached source tree
PROJECT_DIR = Path('/content/art-restoration')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# ── If user requested a refresh, wipe the Drive cache to force re-clone ──────
if REFRESH_FROM_GITHUB and DRIVE_CODE.exists():
    print('REFRESH_FROM_GITHUB=True → wiping Drive code cache to fetch latest')
    shutil.rmtree(DRIVE_CODE)

# ── Restore project code (Drive cache → Colab local) or clone from GitHub ───
if DRIVE_CODE.exists() and (DRIVE_CODE / 'src').exists():
    print(f'Restoring code from Drive cache: {DRIVE_CODE}')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(DRIVE_CODE, PROJECT_DIR)
    print(f'  → {PROJECT_DIR}')
else:
    print('Drive cache empty — cloning private repo from GitHub.')
    print('Generate a Personal Access Token at:')
    print('  https://github.com/settings/tokens  (scope: repo)')
    import getpass
    token = os.environ.get('GITHUB_TOKEN') or getpass.getpass('GitHub token: ')
    repo_url = f'https://{token}@github.com/orivalo/art_restoration.git'
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    !git clone -b phase2-partial-conv {repo_url} {PROJECT_DIR}
    if not (PROJECT_DIR / 'src').exists():
        raise RuntimeError('Clone failed — check your token has `repo` scope.')
    print(f'Caching to Drive: {DRIVE_CODE}')
    if DRIVE_CODE.exists():
        shutil.rmtree(DRIVE_CODE)
    shutil.copytree(
        PROJECT_DIR, DRIVE_CODE,
        ignore=shutil.ignore_patterns('.git', '__pycache__', '*.pyc'),
    )

# Make the package importable
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Show what commit we're on so user can verify they have the right code
!cd {PROJECT_DIR} && git log --oneline -1 2>/dev/null || echo '(commit info unavailable from cache)'

print()
print(f'PROJECT_DIR : {PROJECT_DIR}')
print(f'DRIVE_ROOT  : {DRIVE_ROOT}')
print('Setup complete.')

## Cell 1.5 — Local-Disk Dataset Cache

Drive-mounted file reads are 5-10× slower than reading from Colab's
local SSD because Drive FUSE re-fetches every file from Google Cloud
Storage on demand.  This cell does a one-time zip on Drive, then
unzips to local disk on every session, so each batch loads from the
local SSD instead.

**First session ever**: ~15 min to zip + ~5 min to unzip = ~20 min.
**Every subsequent session**: ~30 s copy + ~5 min unzip = ~5 min.
**After unzip is cached locally**: skipped instantly.

After this, the splits CSVs are rewritten to point at local paths and
saved to `/content/local_splits/`.  Cell 3 picks them up automatically.

In [ ]:
import shutil
import time
import zipfile
import pandas as pd
from pathlib import Path

LOCAL_DATA_ROOT = Path('/content/dataset_local')
DRIVE_PROCESSED = DRIVE_ROOT / 'data' / 'processed'
DRIVE_ZIP       = DRIVE_ROOT / 'data' / 'processed.zip'
LOCAL_ZIP       = Path('/content/processed.zip')
LOCAL_SPLITS    = Path('/content/local_splits')

# ── Step 1: One-time zip creation on Drive (skips if already exists) ────────
if not DRIVE_ZIP.exists():
    print(f'One-time setup: zipping {DRIVE_PROCESSED}')
    print(f'  → {DRIVE_ZIP}')
    print(f'  Takes ~10-15 min for ~4 GB.  Only happens once per Drive.')
    tmp_zip = Path(str(DRIVE_ZIP) + '.tmp')
    t0 = time.time()
    files = sorted(DRIVE_PROCESSED.rglob('*.jpg'))
    with zipfile.ZipFile(tmp_zip, 'w', compression=zipfile.ZIP_STORED) as zf:
        for i, f in enumerate(files, 1):
            arcname = f.relative_to(DRIVE_PROCESSED)
            zf.write(f, arcname)
            if i % 5000 == 0:
                print(f'  {i:,} / {len(files):,} files  ({(time.time()-t0)/60:.1f} min elapsed)')
    tmp_zip.rename(DRIVE_ZIP)  # atomic rename
    print(f'  Done in {(time.time()-t0)/60:.1f} min.  '
          f'Size: {DRIVE_ZIP.stat().st_size/1e9:.2f} GB')
else:
    print(f'Drive zip already exists: {DRIVE_ZIP.name} '
          f'({DRIVE_ZIP.stat().st_size/1e9:.2f} GB)')

# ── Step 2: Per-session copy + unzip to /content/dataset_local/ ─────────────
if not (LOCAL_DATA_ROOT / 'processed').exists():
    print(f'\nCopying zip to local SSD: {LOCAL_ZIP}')
    t0 = time.time()
    shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
    print(f'  Copied in {time.time()-t0:.0f} s')

    print(f'Unzipping → {LOCAL_DATA_ROOT}/processed/')
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, 'r') as zf:
        zf.extractall(LOCAL_DATA_ROOT / 'processed')
    print(f'  Unzipped in {(time.time()-t0)/60:.1f} min')
    LOCAL_ZIP.unlink()  # free up disk
else:
    print(f'Local dataset already cached: {LOCAL_DATA_ROOT}/processed/')

n_local = sum(1 for _ in (LOCAL_DATA_ROOT / 'processed').rglob('*.jpg'))
print(f'  Local image count: {n_local:,}')

# ── Step 3: Rewrite splits CSVs with local paths ────────────────────────────
LOCAL_SPLITS.mkdir(parents=True, exist_ok=True)
DRIVE_PROCESSED_STR = str(DRIVE_PROCESSED)
LOCAL_PROCESSED_STR = str(LOCAL_DATA_ROOT / 'processed')

print(f'\nRewriting splits CSVs with local paths → {LOCAL_SPLITS}/')
for csv_name in ['train.csv', 'val.csv', 'test.csv']:
    src = DRIVE_ROOT / 'data' / 'splits' / csv_name
    dst = LOCAL_SPLITS / csv_name
    df = pd.read_csv(src)
    df['path'] = df['path'].str.replace(DRIVE_PROCESSED_STR, LOCAL_PROCESSED_STR, regex=False)
    df.to_csv(dst, index=False)
    n_exist = df['path'].apply(lambda p: Path(p).exists()).sum()
    status = '✓' if n_exist == len(df) else f'⚠ {len(df) - n_exist} missing'
    print(f'  {csv_name:10s} : {len(df):>6,} rows   {n_exist:>6,} verified  {status}')

print(f'\nLocal data ready.  Cell 3 will pick this up automatically.')

## Cell 2 — GPU Verification

Hard fail if no CUDA is available.  T4 (Colab) or P100 (Kaggle) is mandatory.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU available.\n'
        'Colab : Runtime → Change runtime type → Hardware accelerator → GPU (T4)\n'
        'Kaggle: Settings → Accelerator → GPU P100'
    )

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'cuDNN    : {torch.backends.cudnn.version()}')
print(f'GPU      : {props.name}')
print(f'VRAM     : {vram_gb:.1f} GB')
print(f'SM count : {props.multi_processor_count}')
print()
print('GPU ready.')

## Cell 2.5 — Smoke Test: build all 3 architectures

Run this cell once after `REFRESH_FROM_GITHUB=True` to verify the
refactored code is correct on this machine.  Builds each of the three
architectures (PConv, Vanilla U-Net, Gated U-Net) on a tiny synthetic
batch and confirms the forward pass works end-to-end.

If this cell passes, all three training notebooks (02 / 03 / 04) and the
evaluation notebook (05) will work.  Skip this cell on subsequent runs.

In [ ]:
import torch
from src.models.registry import build_model, SUPPORTED_ARCHS

print('Smoke test: build_model dispatch')
print('━' * 60)

device = torch.device('cuda')
# 256×256 is the minimum size for which all 7 stride-2 encoder stages
# leave enough spatial resolution for the skip connections to align.
B, C, H, W = 1, 3, 256, 256

img  = torch.randn(B, C, H, W, device=device)
mask = (torch.rand(B, 1, H, W, device=device) > 0.4).float()

for arch in SUPPORTED_ARCHS:
    cfg = {'model': {'arch': arch, 'in_channels': 3, 'out_channels': 3, 'verbose': False}}
    model = build_model(cfg).to(device)
    n = sum(p.numel() for p in model.parameters())

    with torch.no_grad():
        output, ret_mask = model(img * mask, mask)

    assert output.shape == (B, C, H, W), f'{arch}: bad output shape {output.shape}'
    assert ret_mask.shape == (B, 1, H, W), f'{arch}: bad mask shape {ret_mask.shape}'
    assert 0.0 <= output.min().item() <= output.max().item() <= 1.0, f'{arch}: output out of [0,1]'

    print(f'  {arch:14s} → {n:>11,} params  forward OK ✓')
    del model
    torch.cuda.empty_cache()

print('\nAll 3 architectures build and run a forward pass correctly.')

## Cell 3 — Config

Load `configs/experiment_configs/pconv_unet.yaml`, override the three
Drive paths so all output ends up on persistent storage, and write a
local patched copy that the Trainer consumes.

The base config already pins paths under
`/content/drive/MyDrive/art_restoration/{checkpoints,logs}/pconv_unet`,
keeping this experiment isolated from the Vanilla and Gated U-Net runs.

**Same-recipe contract:** loss weights, LR, total epochs (20) and seed (42)
are **frozen** identically across `pconv_unet.yaml`, `unet_baseline.yaml`,
and `gated_unet.yaml`.  Only `model.arch` differs.  Edit this cell only
to override per-platform settings (e.g. `batch_size: 8` on Kaggle P100).

In [ ]:
import yaml
from pathlib import Path

CONFIG_PATH       = PROJECT_DIR / 'configs' / 'experiment_configs' / 'pconv_unet.yaml'
LOCAL_CONFIG_PATH = PROJECT_DIR / 'configs' / 'pconv_unet.local.yaml'

with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

# ── Drive-backed paths for checkpoints & logs (Trainer writes here) ─────────
cfg['checkpoint']['dir']  = str(DRIVE_ROOT / 'checkpoints' / cfg['experiment']['name'])
cfg['logging']['log_dir'] = str(DRIVE_ROOT / 'logs'        / cfg['experiment']['name'])

# ── Local-cache splits dir (set by Cell 1.5 — much faster than Drive FUSE) ──
cfg['data']['splits_dir'] = '/content/local_splits'

# ── Per-platform overrides (uncomment as needed) ─────────────────────────────
# cfg['train']['batch_size']  = 8         # 8 on P100, 4 on T4
# cfg['data']['num_workers']  = 4

# ── Persist patched config and display ───────────────────────────────────────
with open(LOCAL_CONFIG_PATH, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Effective configuration')
print('━' * 60)
print(f"Experiment      : {cfg['experiment']['name']}")
print(f"Architecture    : {cfg['model']['arch']}")
print(f"Splits dir      : {cfg['data']['splits_dir']}  (local SSD)")
print(f"Checkpoint dir  : {cfg['checkpoint']['dir']}")
print(f"Log dir         : {cfg['logging']['log_dir']}")
print()
print(f"Batch size      : {cfg['train']['batch_size']}")
print(f"Num epochs      : {cfg['train']['num_epochs']}")
print(f"Learning rate   : {cfg['optim']['lr']}")
print(f"Mixed precision : {cfg['train']['amp']}")
print(f"Mask difficulty : {cfg['data']['difficulty']}")
print()
print(f"Loss weights    : valid={cfg['loss']['lambda_valid']}  "
      f"hole={cfg['loss']['lambda_hole']}  "
      f"perc={cfg['loss']['lambda_perc']}  "
      f"style={cfg['loss']['lambda_style']}  "
      f"tv={cfg['loss']['lambda_tv']}")
print()
print(f'Wrote patched config → {LOCAL_CONFIG_PATH}')

## Cell 4 — Initialize Trainer

Construct the `Trainer`.  It will:

1. Build the PConv U-Net (~25.8 M params)
2. Download VGG-16 weights for the perceptual / style loss (one-time, ~528 MB)
3. Build train + val DataLoaders from the CSV splits
4. **Auto-detect** the latest checkpoint on Drive and resume from it if found

If you see `Resumed at epoch N`, the previous session's state was
recovered.  Otherwise it starts fresh.

In [ ]:
from src.training.trainer import Trainer
from src.utils.checkpoint import find_latest_checkpoint

# Pre-flight: report what we expect Trainer to do
ckpt_dir = Path(cfg['checkpoint']['dir'])
latest = find_latest_checkpoint(ckpt_dir)
if latest is not None:
    print(f'Found checkpoint : {latest.name}')
    print(f'  → will resume from this point')
else:
    print('No checkpoint found in Drive — starting fresh training.')
print()

trainer = Trainer(LOCAL_CONFIG_PATH)

## Cell 5 — Train

Run the full multi-epoch training loop.  Each epoch:

* Train pass with mixed precision + grad clipping
* Validation pass with PSNR / SSIM
* Save `last.pth` (always), `best.pth` (if improved), `epoch_NNN.pth`
  (every `save_every` epochs)
* Append one row to the CSV log
* Save a 5-column visualisation grid to `logs/<exp>/figures/val_epoch_NNN.png`

If Colab disconnects mid-run, just re-execute cells 1 → 5; the Trainer
picks up where it left off.

In [ ]:
trainer.train()

## Cell 6 — Results

Plot the training curves from `training_log.csv` and display the most
recent validation visualisation grid.  This cell is read-only — it can
be run any time during training (in a separate Colab session) to
monitor progress.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# ── Load CSV log ─────────────────────────────────────────────────────────────
log_dir  = Path(cfg['logging']['log_dir'])
csv_path = log_dir / cfg['logging']['csv_filename']
fig_dir  = log_dir / 'figures'

if not csv_path.exists():
    print(f'No training log yet at {csv_path}')
else:
    df = pd.read_csv(csv_path)
    print(f'Loaded {len(df):,} epochs from {csv_path}')

    # ── Plot 4-panel overview ──────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Total training loss
    axes[0, 0].plot(df['epoch'], df['train_total'], color='tab:blue', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Total training loss')
    axes[0, 0].grid(alpha=0.3)

    # Validation PSNR
    axes[0, 1].plot(df['epoch'], df['val_psnr'], color='tab:green', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('PSNR (dB)')
    axes[0, 1].set_title('Validation PSNR')
    axes[0, 1].grid(alpha=0.3)

    # Validation SSIM
    axes[1, 0].plot(df['epoch'], df['val_ssim'], color='tab:purple', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('SSIM')
    axes[1, 0].set_title('Validation SSIM')
    axes[1, 0].grid(alpha=0.3)

    # Loss components on log scale
    component_cols = [c for c in
                      ['train_l1_valid', 'train_l1_hole', 'train_perceptual',
                       'train_style', 'train_tv']
                      if c in df.columns]
    for col in component_cols:
        axes[1, 1].plot(df['epoch'], df[col],
                        label=col.replace('train_', ''), alpha=0.85, linewidth=1.5)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss component')
    axes[1, 1].set_yscale('log')
    axes[1, 1].set_title('Loss components (log)')
    axes[1, 1].grid(alpha=0.3, which='both')
    axes[1, 1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    # ── Print summary ──────────────────────────────────────────────────────
    last = df.iloc[-1]
    best_idx = df['val_psnr'].idxmax()
    best = df.iloc[best_idx]

    print('\nTraining summary')
    print('━' * 60)
    print(f"Last epoch     : {int(last['epoch']):3d}   "
          f"loss={last['train_total']:.3f}   "
          f"PSNR={last['val_psnr']:.2f} dB   "
          f"SSIM={last['val_ssim']:.4f}")
    print(f"Best epoch     : {int(best['epoch']):3d}   "
          f"PSNR={best['val_psnr']:.2f} dB   "
          f"SSIM={best['val_ssim']:.4f}")

# ── Display the most recent validation visualisation ─────────────────────────
if fig_dir.exists():
    figs = sorted(fig_dir.glob('val_epoch_*.png'))
    if figs:
        latest_fig = figs[-1]
        print(f'\nLast validation figure: {latest_fig.name}')
        img = Image.open(latest_fig)
        plt.figure(figsize=(14, 14))
        plt.imshow(img)
        plt.axis('off')
        plt.tight_layout()
        plt.show()